<a href="https://colab.research.google.com/github/wikiwa1/ASD-with-SSL/blob/main/Classifer_with_resnet_and_auroc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

print(os.path.exists('/content/drive'))
print(os.path.exists('/content/drive/MyDrive'))

True
True


In [3]:
# ── imports ──────────────────────────────────────────────────────────────────
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import librosa.display
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from IPython.display import Audio, display

# ── global style ─────────────────────────────────────────────────────────────
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titlesize'] = 11

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── paths ─────────────────────────────────────────────────────────────────────
DATA_ROOT = Path('/content/drive/MyDrive/fan/')   # adjust if data lives elsewhere
MACHINE_IDS = ['id_00', 'id_02', 'id_04', 'id_06']
SR = 16_000               # native sample rate
CHANNEL = 0               # which mic channel to use (0–7); channel 0 is conventional

In [4]:
! [ -e /content ] && pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 140.2 MB/s eta 0:00:00
Mounted at /content/gdrive


In [7]:
logmel_df = pd.read_pickle('/content/drive/MyDrive/fan/6dBlogmel_df.pkl')

In [8]:
df = logmel_df[logmel_df['machine_id'] == 'id_06']

In [9]:
df

,machine_id,label,is_anomaly,logmel
1418,id_02,normal,0,"[[-16.545156, -15.534676, -21.279285, -21.847689, -15.528339, -10.876124, -12.962294, -11.925566, -12.149639, -15.20974, -18.227577, -19.4579, -13.485746, -16.960777, -20.95616, -20.205751, -17.454773, -13.708675, -14.525463, -20.223068, -15.006531, -13.511448, -15.531536, -16.56918, -14.201122, -14.63496, -15.66325, -21.863876, -16.47874, -11.940105, -11.9264145, -14.08823, -16.284164, -16.710796, -16.044731, -15.073612, -15.1292305, -20.097717, -17.978008, -17.363205, -15.05761, -11.471962, -13.0291195, -19.114119, -20.126125, -14.792141, -13.970896, -16.910921, -19.521639, -17.037434, -..."
1419,id_02,normal,0,"[[-24.034477, -24.72548, -25.43589, -20.77903, -21.841667, -29.944935, -30.952034, -32.99755, -26.21608, -25.027153, -22.821404, -27.434574, -22.87648, -23.603848, -28.431705, -29.056492, -24.10038, -19.566723, -17.405972, -22.092106, -22.251846, -20.299805, -19.658665, -20.397339, -25.500526, -28.760487, -30.86166, -33.53853, -31.47784, -25.6293, -24.28181, -27.674564, -33.319183, -27.974648, -25.134907, -28.694145, -25.683697, -21.824303, -26.009003, -28.779675, -28.231125, -22.33183, -22.69994, -26.732685, -19.70958, -21.06364, -25.242477, -26.36713, -23.32137, -21.545757, -22.161972, -..."
1420,id_02,normal,0,"[[-24.380066, -23.373833, -22.641106, -21.268059, -23.53265, -20.71077, -21.02428, -24.36206, -21.346668, -23.772556, -22.495148, -20.277218, -19.919209, -24.393665, -27.744995, -24.323017, -24.587208, -24.925465, -26.283363, -27.38058, -24.861137, -21.914497, -22.855648, -22.35347, -21.745514, -21.278355, -23.12788, -20.976524, -17.888119, -17.651432, -19.087128, -19.733849, -21.002686, -26.342392, -25.086193, -23.556767, -23.053516, -21.932938, -20.303715, -21.718647, -20.392883, -21.9822, -23.257217, -24.828838, -27.39009, -28.87912, -26.034046, -23.820972, -16.595997, -14.280705, -20.2..."
1421,id_02,normal,0,"[[-13.8466015, -13.453873, -18.130762, -25.01385, -19.917255, -14.820633, -13.379425, -13.310631, -14.9965, -12.884745, -12.701998, -16.924519, -20.87746, -16.15067, -17.601547, -20.20421, -15.350588, -12.75057, -13.530729, -17.143776, -17.925926, -17.557297, -12.847467, -12.986488, -14.154171, -17.627869, -16.364761, -18.272131, -16.286222, -14.7456, -15.378563, -12.860413, -15.574802, -17.418274, -19.676746, -19.269974, -22.36477, -27.612679, -28.481731, -19.072784, -14.650444, -14.753593, -17.845615, -19.020336, -18.340767, -18.901403, -20.913012, -19.100395, -17.43535, -19.241673, -20...."
1422,id_02,normal,0,"[[-26.85623, -29.31844, -26.898346, -21.111275, -23.60522, -35.249443, -29.448662, -24.73746, -24.291739, -21.8917, -23.710363, -26.007889, -23.78119, -25.949171, -28.525505, -25.41008, -23.103834, -22.355556, -24.455925, -20.051878, -23.3652, -31.484325, -29.809372, -28.85395, -27.727907, -27.987232, -33.199875, -25.826239, -25.423996, -26.993145, -27.310059, -23.92225, -29.12347, -28.164055, -25.558205, -25.324215, -26.414757, -23.262867, -24.328987, -26.266209, -32.436802, -29.510387, -22.167088, -23.862267, -22.742096, -21.422646, -21.630419, -22.801052, -27.686342, -28.623379, -36.213..."
...,...,...,...,...
2788,id_02,abnormal,1,"[[-21.322021, -21.522799, -19.123882, -20.679691, -16.180977, -17.608524, -23.967524, -23.928978, -20.504488, -20.197395, -19.46155, -15.678757, -14.367329, -17.352058, -24.388657, -22.886223, -19.32417, -21.314568, -20.04267, -18.491766, -19.198883, -28.77089, -22.986126, -16.748459, -16.635563, -18.274881, -21.10887, -21.689598, -18.739021, -20.959372, -22.761606, -21.04179, -22.257654, -14.670719, -15.554462, -19.563238, -22.24562, -21.21897, -20.821527, -22.44939, -23.618605, -18.880615, -23.224833, -25.193008, -20.840418, -18.10262, -21.352844, -26.488575, -21.056889, -15.853464, -16...."
2789,id_02,abnormal,1,"[[-22.058025, -16.790031, -14.164246, -11.852001, -10.81092, -10.476733, -13.196003, -13.609394, -16.512383, -14.244896, -14.724087, -14.031061, -15.974209, -9.233905, -10.170576, -9.7984

In [ ]:
print(df.columns)

Index(['machine_id', 'label', 'is_anomaly', 'logmel'], dtype='object')


In [9]:
from sklearn.model_selection import train_test_split

train_val_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

In [10]:
train_val_df

,machine_id,label,is_anomaly,logmel
4302,id_06,normal,0,"[[-24.142298, -18.56737, -15.2427845, -15.988735, -24.37142, -25.107542, -21.612835, -18.550694, -19.278286, -23.44798, -23.048985, -17.623152, -17.89921, -19.89073, -19.431112, -23.454317, -26.5911, -20.533134, -16.76111, -18.356413, -19.853315, -24.206503, -25.375845, -21.878662, -24.256975, -23.655874, -23.581963, -20.197409, -16.812937, -18.189035, -20.742914, -19.72168, -18.563864, -18.506887, -23.122978, -19.578932, -17.624413, -20.800306, -31.035593, -30.229708, -22.843393, -21.262533, -17.88798, -20.924707, -19.602133, -18.505022, -20.061714, -22.016272, -22.256693, -20.788183, -22..."
4676,id_06,normal,0,"[[-24.448587, -18.909004, -22.292723, -22.46579, -20.663158, -16.509, -18.208809, -17.93164, -11.786886, -12.889287, -18.227345, -15.704852, -17.75616, -20.43827, -15.611222, -15.457746, -16.05245, -18.455578, -18.420595, -22.962687, -20.7086, -17.846355, -17.3467, -17.524834, -20.551353, -17.014511, -17.925209, -17.039604, -20.255295, -18.866095, -16.659178, -15.592948, -20.593487, -20.17605, -17.141409, -17.490204, -21.99923, -19.358776, -16.80726, -17.629267, -16.891418, -17.160128, -23.268402, -23.894894, -18.867067, -18.406673, -17.549286, -14.108775, -17.3673, -25.519365, -24.631357,..."
4277,id_06,normal,0,"[[-20.76307, -14.790444, -12.856534, -15.099235, -20.775425, -27.985634, -23.980093, -18.19113, -16.911634, -21.201555, -22.666315, -18.021526, -13.094672, -14.386182, -17.171024, -17.509716, -20.132772, -19.579166, -13.754119, -14.443234, -16.526854, -17.825983, -16.089186, -19.307165, -14.467502, -14.7472925, -19.891872, -18.206318, -21.82431, -24.287233, -15.80763, -12.319187, -14.084937, -19.129633, -18.12827, -17.096033, -19.526514, -20.123615, -15.879254, -15.683616, -18.200048, -22.031513, -17.330896, -13.972491, -15.888268, -23.181852, -20.02631, -17.578653, -16.353436, -19.989582,..."
4839,id_06,normal,0,"[[-20.427097, -16.375065, -16.623835, -15.9851675, -20.999357, -21.49868, -22.281904, -17.731623, -15.148913, -17.017225, -20.430136, -15.847299, -18.120346, -19.773544, -19.746819, -18.497791, -16.243446, -15.449405, -19.328005, -19.828518, -17.07891, -16.585703, -21.946077, -16.215237, -17.782015, -22.3386, -18.40297, -14.007609, -17.179167, -18.450327, -16.84589, -20.480137, -21.510775, -23.417196, -25.12237, -26.236757, -22.21368, -19.339594, -18.815672, -17.308037, -20.881721, -25.673555, -22.324156, -21.125706, -21.719505, -24.66987, -18.748247, -19.051237, -15.429357, -11.256206, -1..."
4517,id_06,normal,0,"[[-24.114864, -25.584675, -30.11332, -31.90527, -28.54768, -26.7476, -25.53109, -25.936344, -27.35896, -29.439274, -26.040306, -24.445557, -29.588432, -29.845514, -34.67688, -33.152096, -31.457336, -32.424976, -30.725006, -27.784498, -28.299877, -28.175022, -25.846516, -24.922157, -28.91309, -27.832611, -26.747108, -28.895683, -24.642746, -25.736807, -27.122675, -22.72228, -23.601269, -29.684479, -34.46354, -26.821407, -23.728298, -30.15657, -32.68706, -32.336124, -29.595335, -28.459421, -30.425152, -32.40405, -22.92627, -16.589722, -14.669462, -16.86193, -15.92672, -18.205162, -22.177177,..."
...,...,...,...,...
5431,id_06,abnormal,1,"[[-18.541054, -11.351047, -13.338771, -15.560472, -15.802562, -18.818115, -18.817898, -14.631951, -12.990575, -13.956454, -13.552926, -13.945127, -13.490882, -15.097323, -11.060803, -13.682916, -21.268593, -14.52465, -11.105924, -12.466811, -11.959083, -11.289952, -16.113045, -11.391894, -12.513452, -16.666359, -17.353664, -14.243747, -11.817658, -11.034768, -10.994981, -16.520004, -12.390273, -11.597133, -9.516789, -12.118428, -11.764047, -12.211442, -14.119655, -11.028695, -10.2486315, -14.492154, -13.995612, -16.282707, -16.730835, -15.390376, -14.3272085, -15.055934, -13.174912, -12.19..."
4486,id_06,normal,0,"[[-23.55597, -22.535154, -16.769638, -13.537421, -13.576987, -14.426632, -18.277836, -18.039877, -15.677778, -13.113239, -12.642869, -17.639753, -18.200506, -21.233446, -20.58901, -17.38527

In [11]:
import torch
from fastai.vision.all import *
from fastai.data.transforms import IndexSplitter



def get_x(row):
    arr = row['logmel'].astype('float32')      # ensure float32
    arr = np.expand_dims(arr, axis=0)           # add channel dim: (1, n_mels, n_frames)
    return torch.from_numpy(arr)

def get_y(row):
    return row['is_anomaly']
def TensorBlock():
    return TransformBlock()

In [13]:
from sklearn.model_selection import train_test_split
from fastai.torch_core import TensorImage, TensorMultiCategory
import torch.nn as nn
from torchvision import models
from fastai.metrics import RocAucBinary, accuracy

def build_model(num_classes=2):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    # Swap first conv layer: 3 channels → 1 channel
    # We average the pretrained RGB weights instead of reinitializing from scratch
    old_conv = model.conv1
    new_conv = nn.Conv2d(
        in_channels=1,
        out_channels=old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=(old_conv.bias is not None),
    )
    with torch.no_grad():
        new_conv.weight[:] = old_conv.weight.mean(dim=1, keepdim=True)
    model.conv1 = new_conv

    # Swap final FC layer for your number of classes
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    return model

In [15]:
# Create train/test split
train_df, test_df = train_test_split(
df,
test_size=0.2,
stratify=df['is_anomaly'],
random_state=7
)

combined = pd.concat([train_df, test_df]).reset_index(drop=True)

valid_idx = list(range(len(train_df), len(combined)))

dblock = DataBlock( blocks=(TensorBlock, CategoryBlock), get_x=get_x, get_y=get_y, splitter=IndexSplitter(valid_idx) )

dls = dblock.dataloaders(combined, bs=64)

    # Model
model = build_model(num_classes=2)

    # Train
learn = Learner(
dls,
model,
loss_func=CrossEntropyLossFlat(),
metrics=[
    accuracy,
    RocAucBinary()
]
)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 242MB/s]


In [16]:
learn.fine_tune(10)

epoch,train_loss,valid_loss,accuracy,roc_auc_score,time
0,0.140309,0.198091,0.945652,0.999455,00:02


epoch,train_loss,valid_loss,accuracy,roc_auc_score,time
0,0.033899,0.401274,0.938406,1.000000,00:01
1,0.015686,0.000046,1.000000,1.000000,00:01
2,0.008653,0.000046,1.000000,1.000000,00:01
3,0.005306,0.000038,1.000000,1.000000,00:01
4,0.030232,0.570723,0.757246,0.893382,00:01
5,0.023074,0.096958,0.971014,1.000000,00:01
6,0.015969,0.000452,1.000000,1.000000,00:01
7,0.011241,0.000081,1.000000,1.000000,00:01
8,0.007877,0.000094,1.000000,1.000000,00:01
9,0.005668,0.000109,1.000000,1.000000,00:01


In [17]:
val_loss, acc, auroc = learn.validate()
print(val_loss, acc, auroc)

0.00010856192238861695 1.0 1.0


In [21]:
test_dblock = DataBlock(
    blocks=(TensorBlock, CategoryBlock),
    get_x=get_x,
    get_y=get_y
)

test_dls = test_dblock.dataloaders(
    test_df,
    bs=64,
    shuffle=False
)

test_dl = test_dls.train

In [22]:
preds, targets = learn.get_preds(dl=test_dl)

In [23]:
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix

anomaly_scores = preds[:, 1]

test_auroc = roc_auc_score(
    targets.numpy(),
    anomaly_scores.numpy()
)

test_accuracy = accuracy_score(
    targets.numpy(),
    preds.argmax(dim=1).numpy()
)

cm = confusion_matrix(
    targets.numpy(),
    preds.argmax(dim=1).numpy()
)

print(f"Test AUROC: {test_auroc:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(cm)

Test AUROC: 1.0000
Test Accuracy: 1.0000
[[166   0]
 [  0  55]]
